### Tools

Model can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairing of:
    
    
    
    1. A schema, including the name of the tool, a description and/or argument definations (often a JSON schema).
    2. A function or coroutine to execute.

In [2]:
import os
from dotenv import load_dotenv
from langchain.chat_models import  init_chat_model
load_dotenv()

os.environ["GoogleAPIKey"] = os.getenv("googleAIStudioKey")
os.environ["GROQ_API_KEY"] = os.getenv("groqAPIKey")

model = init_chat_model("groq:qwen/qwen3-32b")

# how to invoke a model
response = model.invoke("Why do parrots talk?")
response


AIMessage(content='<think>\nOkay, so I need to figure out why parrots talk. Let me start by recalling what I know. Parrots are known for their ability to mimic human speech, right? But why do they do that in the first place? Maybe it\'s something to do with communication. I remember that in the wild, parrots might use sounds to interact with each other, like calling to their mates or warning others in their flock. But when they\'re in captivity, they start talking to humans. Is that because they adapt to their environment?\n\nI\'ve heard that parrots have a strong social structure. They form bonds with their flock members. If they\'re kept as pets, maybe they treat their owners as part of their flock. So, talking could be a way to bond or communicate with their humans. But why speech specifically? Do they enjoy it, or is it a learned behavior?\n\nI think some parrots can learn to say words through repetition. The owner might say something, and the parrot repeats it. But does the parrot

In [4]:
from langchain_community.tools import tool


# Using tool decoratore while creating a tool

@tool
def get_weather(location):
    """Get the weather of a location"""
    return f"It's sunny in {location}"


### Binding the tool with the model

model_with_tools = model.bind_tools([get_weather])

response = model_with_tools.invoke("Whats the weather in Pune")

print(response)


content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Pune. I need to use the get_weather function. The function requires a location parameter. Pune is a city in India, so I should specify that as the location. I\'ll call the function with "Pune" as the argument. Let me make sure there are no spelling mistakes. Everything looks good. Time to format the tool call correctly.\n', 'tool_calls': [{'id': '6pn019amn', 'function': {'arguments': '{"location":"Pune"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 105, 'prompt_tokens': 146, 'total_tokens': 251, 'completion_time': 0.176903714, 'completion_tokens_details': {'reasoning_tokens': 80}, 'prompt_time': 0.005699561, 'prompt_tokens_details': None, 'queue_time': 0.049141868, 'total_time': 0.182603275}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': 

### Tool Execution Loops

In [9]:
# Step1: Model generates tool calls
messages = [{"role":"user", "content":"What's the weather in Bosten"}]

ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step2Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append (tool_result)
   
 # Step 3 : Pass results back to model for final response
final_response =  model_with_tools.invoke(messages)
print(final_response.text)

The weather in Bosten is currently sunny. Let me know if you need more details! ☀️
